# IMDb Review Ingestion

This notebook ingests the six IMDb review JSON files into Parquet format for the raw data layer of the project. The files are processed in batches so the full dataset does not need to be loaded into memory at once.

## 1. Set up project and source paths

In [2]:
from pathlib import Path

# The notebooks folder is inside the main project folder
project_root = Path.cwd().parent

# The original IMDb files are stored outside the project
source_folder = Path(r"C:\Data\imdb_reviews")

# Save the ingested IMDb data in the project's raw data folder
output_folder = project_root / "data" / "raw" / "imdb_reviews"

# Create the output folder if it does not already exist
output_folder.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Source folder:", source_folder)
print("Output folder:", output_folder)

Project root: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project
Source folder: C:\Data\imdb_reviews
Output folder: C:\Users\dorca\OneDrive\Desktop\movies-tv-data-project\data\raw\imdb_reviews


## 2. Ingest IMDb reviews to Parquet

The source data is split across six large JSON files. Each file is streamed and written to Parquet in batches of 50,000 reviews to keep memory usage manageable.

In [3]:
import ijson
import pyarrow as pa
import pyarrow.parquet as pq

# All six files that make up the IMDb review dataset
files_to_convert = [
    "part-01.json",
    "part-02.json",
    "part-03.json",
    "part-04.json",
    "part-05.json",
    "part-06.json"
]

# Read and write the reviews in batches so the full files are never loaded into memory
batch_size = 50000

for file_name in files_to_convert:

    source_file = source_folder / file_name
    output_file = output_folder / file_name.replace(".json", ".parquet")

    rows = []
    writer = None
    written = 0

    print(f"\nStarting {file_name}")

    with open(source_file, "rb") as f:
        reviews = ijson.items(f, "item")

        for review in reviews:
            rows.append(review)

            # Once 50,000 reviews are collected, write that batch to Parquet
            if len(rows) == batch_size:
                table = pa.Table.from_pylist(rows)

                # Use the first batch to set the schema for this Parquet file
                if writer is None:
                    writer = pq.ParquetWriter(output_file, table.schema)

                writer.write_table(table)

                written += len(rows)
                rows = []

                print(f"{written:,} reviews written")

    # Write any reviews left after the last full batch
    if rows:
        table = pa.Table.from_pylist(rows)

        if writer is None:
            writer = pq.ParquetWriter(output_file, table.schema)

        writer.write_table(table)
        written += len(rows)

    # Close the Parquet file after all reviews have been written
    if writer is not None:
        writer.close()

    # Check the finished file to make sure the row count matches what was written
    metadata = pq.read_metadata(output_file)

    print(f"Finished {file_name}")
    print(f"Rows written: {written:,}")
    print(f"Rows in Parquet: {metadata.num_rows:,}")
    print(f"Columns: {metadata.num_columns}")


Starting part-01.json
50,000 reviews written
100,000 reviews written
150,000 reviews written
200,000 reviews written
250,000 reviews written
300,000 reviews written
350,000 reviews written
400,000 reviews written
450,000 reviews written
500,000 reviews written
550,000 reviews written
600,000 reviews written
650,000 reviews written
700,000 reviews written
750,000 reviews written
800,000 reviews written
850,000 reviews written
900,000 reviews written
950,000 reviews written
1,000,000 reviews written
Finished part-01.json
Rows written: 1,010,293
Rows in Parquet: 1,010,293
Columns: 9

Starting part-02.json
50,000 reviews written
100,000 reviews written
150,000 reviews written
200,000 reviews written
250,000 reviews written
300,000 reviews written
350,000 reviews written
400,000 reviews written
450,000 reviews written
500,000 reviews written
550,000 reviews written
600,000 reviews written
650,000 reviews written
700,000 reviews written
750,000 reviews written
800,000 reviews written
850,00

## 3. Validate the ingested dataset

In [4]:
import pyarrow.parquet as pq
from pathlib import Path

# Get all of the IMDb Parquet files created during ingestion
parquet_files = sorted(output_folder.glob("part-*.parquet"))

total_rows = 0
schemas = []
total_parquet_size = 0

for file in parquet_files:
    metadata = pq.read_metadata(file)

    total_rows += metadata.num_rows
    total_parquet_size += file.stat().st_size

    # Save each file's schema so we can check that they all match
    schemas.append(pq.read_schema(file))

    print(
        f"{file.name}: "
        f"{metadata.num_rows:,} rows, "
        f"{metadata.num_columns} columns"
    )

# Check whether every Parquet file has the same schema
schemas_match = all(schema == schemas[0] for schema in schemas)

print("\nTOTAL ROWS:", f"{total_rows:,}")
print("ALL SCHEMAS MATCH:", schemas_match)
print(
    "TOTAL PARQUET SIZE:",
    f"{total_parquet_size / (1024**3):.2f} GB"
)

part-01.parquet: 1,010,293 rows, 9 columns
part-02.parquet: 1,012,212 rows, 9 columns
part-03.parquet: 1,015,000 rows, 9 columns
part-04.parquet: 1,019,000 rows, 9 columns
part-05.parquet: 1,014,997 rows, 9 columns
part-06.parquet: 499,997 rows, 9 columns

TOTAL ROWS: 5,571,499
ALL SCHEMAS MATCH: True
TOTAL PARQUET SIZE: 3.94 GB


In [5]:
# Get the six original IMDb JSON files
json_files = sorted(source_folder.glob("part-*.json"))

total_json_size = sum(file.stat().st_size for file in json_files)
total_parquet_size = sum(file.stat().st_size for file in parquet_files)

print(
    "TOTAL JSON SIZE:",
    f"{total_json_size / (1024**3):.2f} GB"
)

print(
    "TOTAL PARQUET SIZE:",
    f"{total_parquet_size / (1024**3):.2f} GB"
)

print(
    "PARQUET AS % OF JSON:",
    f"{total_parquet_size / total_json_size:.2%}"
)

TOTAL JSON SIZE: 7.09 GB
TOTAL PARQUET SIZE: 3.94 GB
PARQUET AS % OF JSON: 55.48%


In [6]:
# The source profiling showed that the full IMDb dataset contains 5,571,499 reviews
expected_rows = 5_571_499

# Stop the notebook if the ingested row count does not match the source
assert total_rows == expected_rows, (
    f"Expected {expected_rows:,} rows but found {total_rows:,}"
)

# Stop the notebook if the Parquet files do not all have the same schema
assert schemas_match, "The Parquet files do not all have the same schema"

print("IMDb ingestion validation passed.")

IMDb ingestion validation passed.


## 4. Ingestion result

The six IMDb JSON source files were successfully ingested into six Parquet files in the raw data layer.

- Total reviews: 5,571,499
- Columns: 9
- Original JSON size: 7.09 GB
- Parquet size: 3.94 GB
- Parquet size relative to JSON: 55.48%
- All Parquet files use the same schema